# DenseNet201 com transfer learning

Fine-tuning completo de uma DenseNet201 pré-treinada na ImageNet para as 38 classes do PlantVillage. Treinamento, checkpoint e métricas usam somente `train` e `validation`. **O conjunto de teste permanece reservado e não é iterado.**

## Caminhos e ambiente

A configuração funciona localmente e no Google Colab. Ajuste `BASE_DIR` apenas se o repositório estiver em outro caminho.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MOUNT_DRIVE = False
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    if IN_COLAB:
        candidates.append(Path('/content/tcc-plant-disease-classification'))
    for candidate in candidates:
        if (candidate / 'src' / 'train_densenet201.py').exists():
            return candidate
    return Path('/content/tcc-plant-disease-classification') if IN_COLAB else current

BASE_DIR = find_project_root()
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
DENSENET201_RESULTS_DIR = RESULTS_DIR / 'densenet201'
SRC_DIR = BASE_DIR / 'src'
ZIP_PATH = DATA_DIR / 'data.zip'
LEAF_MAP_PATH = DATA_DIR / 'leaf_grouping' / 'leaf-map.json'
RAW_METADATA_CSV = RESULTS_DIR / 'plantvillage_metadata_raw_color.csv'
METADATA_CSV = RESULTS_DIR / 'plantvillage_metadata_split.csv'
IMAGE_ROOT = Path('/content/plantvillage_color') if IN_COLAB else DATA_DIR / 'plantvillage_color'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DENSENET201_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Diretório src não encontrado: {SRC_DIR}')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Base:', BASE_DIR)
print('Resultados da DenseNet201:', DENSENET201_RESULTS_DIR)
print('Imagens:', IMAGE_ROOT)

## Dependências

In [ ]:
required_packages = {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
}
missing_packages = [
    package for module_name, package in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

import json
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display

## Protocolo experimental

In [ ]:
SEED = 42
NUM_CLASSES = 38
BATCH_SIZE = 32
MAX_EPOCHS = 30
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-4
NUM_WORKERS = 2 if IN_COLAB else 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo selecionado automaticamente:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from densenet201_model import (
    DEFAULT_WEIGHTS,
    create_densenet201,
    densenet201_weights_name,
    validate_densenet201_preprocessing,
)
from plantvillage_audit import build_metadata_dataframe, save_metadata_csv
from plantvillage_pytorch import build_image_transforms, extract_raw_color_from_zip
from plantvillage_split import save_split_metadata_csv, split_metadata_by_leaf_id
from train_densenet201 import train_densenet201

## Dados e splits existentes

As rotinas são as mesmas dos modelos anteriores. O CSV de split existente é reutilizado sem alterações.

In [ ]:
DOWNLOAD_DATA_IF_MISSING = True
EXTRACT_IMAGES_IF_MISSING = True

if DOWNLOAD_DATA_IF_MISSING and (not ZIP_PATH.exists() or not LEAF_MAP_PATH.exists()):
    if importlib.util.find_spec('huggingface_hub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
    from huggingface_hub import hf_hub_download
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        ZIP_PATH = Path(hf_hub_download(
            repo_id='mohanty/PlantVillage', filename='data.zip',
            repo_type='dataset', local_dir=str(DATA_DIR),
        ))
    if not LEAF_MAP_PATH.exists():
        LEAF_MAP_PATH = Path(hf_hub_download(
            repo_id='mohanty/PlantVillage', filename='leaf_grouping/leaf-map.json',
            repo_type='dataset', local_dir=str(DATA_DIR),
        ))

if not METADATA_CSV.exists():
    if not ZIP_PATH.exists() or not LEAF_MAP_PATH.exists():
        raise FileNotFoundError('Arquivos necessários para gerar os splits não encontrados.')
    metadata = build_metadata_dataframe(ZIP_PATH, LEAF_MAP_PATH)
    save_metadata_csv(metadata, RAW_METADATA_CSV)
    metadata_split = split_metadata_by_leaf_id(metadata)
    save_split_metadata_csv(metadata_split, METADATA_CSV)
else:
    print('CSV de split existente reutilizado:', METADATA_CSV)

image_root_has_files = IMAGE_ROOT.is_dir() and any(IMAGE_ROOT.iterdir())
if EXTRACT_IMAGES_IF_MISSING and not image_root_has_files:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'ZIP do PlantVillage não encontrado: {ZIP_PATH}')
    extraction = extract_raw_color_from_zip(ZIP_PATH, IMAGE_ROOT)
    print('Extração:', extraction)
elif not image_root_has_files:
    raise FileNotFoundError(f'Diretório de imagens vazio ou ausente: {IMAGE_ROOT}')

## Conferência do protocolo e preprocessing

A contagem consulta somente os metadados; nenhuma imagem de teste é carregada. A rotina também comprova que a normalização global é compatível com os pesos da DenseNet201.

In [ ]:
metadata_df = pd.read_csv(METADATA_CSV)
split_sizes = metadata_df['split'].value_counts().rename_axis('split').reset_index(name='num_images')
display(split_sizes)

preprocessing = validate_densenet201_preprocessing(DEFAULT_WEIGHTS)
print('Pesos:', densenet201_weights_name(DEFAULT_WEIGHTS))
print('Tamanho da entrada:', (224, 224))
print('Compatibilidade da normalização:', preprocessing)
print('Transform de treino:', build_image_transforms('train'))
print('Transform de validação:', build_image_transforms('validation'))
print('Split usado nesta etapa: validation')
print('Split reservado: test (DataLoader não iterado)')

## Verificação do modelo pré-treinado

Somente `classifier` é substituída. Todos os parâmetros permanecem treináveis.

In [ ]:
preview_model = create_densenet201(num_classes=NUM_CLASSES)
total_parameters = sum(parameter.numel() for parameter in preview_model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in preview_model.parameters() if parameter.requires_grad
)
print('Classe:', type(preview_model).__name__)
print('Classifier final:', preview_model.classifier)
print('Parâmetros totais:', f'{total_parameters:,}')
print('Parâmetros treináveis:', f'{trainable_parameters:,}')
assert preview_model.classifier.out_features == NUM_CLASSES
assert total_parameters == trainable_parameters
del preview_model

## Treinamento e avaliação na validação

A execução manual usa Adam, `CrossEntropyLoss`, learning rate `1e-4`, até 30 épocas e early stopping por `validation_loss`. O melhor checkpoint é restaurado antes da inferência.

In [ ]:
model, history, checkpoint_path = train_densenet201(
    metadata_csv=METADATA_CSV,
    image_root=IMAGE_ROOT,
    output_dir=DENSENET201_RESULTS_DIR,
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
    num_classes=NUM_CLASSES,
    num_workers=NUM_WORKERS,
    seed=SEED,
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=MIN_DELTA,
)
print('Melhor checkpoint restaurado:', checkpoint_path)

## Arquitetura e complexidade

In [ ]:
with (DENSENET201_RESULTS_DIR / 'architecture_summary.json').open(encoding='utf-8') as file:
    architecture = json.load(file)
display(pd.Series({
    'model_class': architecture['model_class'],
    'architecture': architecture['architecture'],
    'pretrained_weights': architecture['pretrained_weights'],
    'total_parameters': architecture['total_parameters'],
    'trainable_parameters': architecture['trainable_parameters'],
    'final_layer_in_features': architecture['final_layer_in_features'],
    'num_output_classes': architecture['num_output_classes'],
    'final_layer_parameter_count': architecture['final_layer_parameter_count'],
}, name='architecture').to_frame())
display(pd.DataFrame(architecture['main_blocks']['dense_blocks']))

## Histórico, curvas e tempos

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['validation_loss'], label='validation')
axes[0].set(title='Loss', xlabel='Época', ylabel='CrossEntropyLoss')
axes[0].legend()
axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='train')
axes[1].plot(history_df['epoch'], history_df['validation_accuracy'], label='validation')
axes[1].set(title='Accuracy', xlabel='Época', ylabel='Accuracy')
axes[1].legend()
axes[2].bar(history_df['epoch'], history_df['epoch_time_seconds'])
axes[2].set(title='Tempo por época', xlabel='Época', ylabel='Segundos')
for axis in axes:
    axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

with (DENSENET201_RESULTS_DIR / 'timing_summary.json').open(encoding='utf-8') as file:
    timing_summary = json.load(file)
display(pd.Series(timing_summary['training'], name='training').to_frame())
display(pd.Series(timing_summary['validation_inference'], name='validation_inference').to_frame())

## Métricas gerais e por classe

In [ ]:
validation_metrics_df = pd.read_csv(DENSENET201_RESULTS_DIR / 'validation_metrics.csv')
class_metrics_df = pd.read_csv(DENSENET201_RESULTS_DIR / 'validation_metrics_by_class.csv')
display(validation_metrics_df.T.rename(columns={0: 'value'}))
display(class_metrics_df)

## Matriz de confusão da validação

In [ ]:
confusion_matrix_path = DENSENET201_RESULTS_DIR / 'validation_confusion_matrix.png'
display(Image(filename=str(confusion_matrix_path), width=1100))

## Artefatos gerados

In [ ]:
artifacts = sorted(path for path in DENSENET201_RESULTS_DIR.iterdir() if path.is_file())
display(pd.DataFrame({
    'arquivo': [path.name for path in artifacts],
    'caminho': [str(path) for path in artifacts],
    'tamanho_bytes': [path.stat().st_size for path in artifacts],
}))
with (DENSENET201_RESULTS_DIR / 'experiment_metadata.json').open(encoding='utf-8') as file:
    experiment_metadata = json.load(file)
display(pd.Series(experiment_metadata, name='experiment_metadata').to_frame())
print('Todos os resultados da DenseNet201 estão em:', DENSENET201_RESULTS_DIR)